# Cleanup

This notebook deletes resources created via API (Registry, records, guardrail). The CFN stack deletion handles everything else.

> **If continuing to Module 4**, only delete the guardrail and Registry resources. Do NOT delete the CFN stack.

In [ ]:
%pip install "boto3==1.42.87" --quiet
import boto3
assert tuple(int(x) for x in boto3.__version__.split(".")[:3]) >= (1, 42, 87), f"boto3 too old: {boto3.__version__} (need >=1.42.87)"
print(f"boto3 {boto3.__version__} OK")

## Step 1: Setup

In [ ]:
import boto3, json

region = boto3.session.Session().region_name or "us-west-2"
bedrock = boto3.client("bedrock", region_name=region)
lam = boto3.client("lambda", region_name=region)

# Find registry
cp = boto3.client("bedrock-agentcore-control", region_name=region)
registries = cp.list_registries().get("registries", [])
workshop_reg = [r for r in registries if r["name"] == "workshop-registry"]

REGISTRY_ID = (workshop_reg[0].get("registryId") or workshop_reg[0]["registryArn"].split("/")[-1]) if workshop_reg else None
print(f"Registry ID: {REGISTRY_ID or 'Not found (already deleted?)'}")

## Step 2: Delete Bedrock Guardrail

In [ ]:
GUARDRAIL_NAME = "workshop-tool-output-guardrail"
existing = bedrock.list_guardrails(maxResults=100).get("guardrails", [])
match = [g for g in existing if g["name"] == GUARDRAIL_NAME]

if match:
    bedrock.delete_guardrail(guardrailIdentifier=match[0]["id"])
    print(f"Deleted guardrail: {match[0]['id']}")
else:
    print("Guardrail not found (already deleted)")

# Clear guardrail env var from response interceptor
try:
    fn = "ac-gateway-response-interceptor"
    config = lam.get_function_configuration(FunctionName=fn)
    env_vars = config.get("Environment", {}).get("Variables", {})
    env_vars["BEDROCK_GUARDRAIL_ID"] = ""
    lam.update_function_configuration(FunctionName=fn, Environment={"Variables": env_vars})
    print(f"Cleared guardrail ID from {fn}")
except Exception as e:
    print(f"Note: {e}")

## Step 2b: Delete Policy Engine and Cedar Policies

In [ ]:
# Delete Policy Engine. The engine cannot be deleted while it is still
# associated with any Gateway, so detach it from EVERY gateway that references
# it (a full workshop run has several: ac-tools-gateway, tools-gateway, and the
# FAST gateway) before deleting its policies and the engine itself.
ENGINE_NAME = "workshop_policy_engine"
engines = cp.list_policy_engines().get("policyEngines", [])
engine_match = [e for e in engines if e.get("name") == ENGINE_NAME]

if engine_match:
    engine_id = engine_match[0]["policyEngineId"]
    # Detach from every gateway that has a policy-engine configuration.
    for gw in cp.list_gateways().get("items", []):
        gw_id = gw.get("gatewayId")
        try:
            full = cp.get_gateway(gatewayIdentifier=gw_id)
            if full.get("policyEngineConfiguration"):
                cp.update_gateway(
                    gatewayIdentifier=gw_id,
                    name=full["name"],
                    roleArn=full["roleArn"],
                    protocolType=full["protocolType"],
                    authorizerType=full["authorizerType"],
                    authorizerConfiguration=full["authorizerConfiguration"],
                )
                print(f"  Detached Policy Engine from gateway {full.get('name', gw_id)}")
        except Exception as e:
            print(f"  Detach skipped for {gw_id}: {e}")
    # Delete policies first
    for p in cp.list_policies(policyEngineId=engine_id).get("policies", []):
        try:
            cp.delete_policy(policyEngineId=engine_id, policyId=p["policyId"])
            print(f"  Deleted policy: {p.get('name', p['policyId'])}")
        except Exception as e:
            print(f"  Error deleting policy: {e}")
    # Delete engine — tolerate a lingering association rather than aborting the
    # whole cleanup notebook; the engine is also removed with the CFN stack.
    try:
        cp.delete_policy_engine(policyEngineId=engine_id)
        print(f"Deleted Policy Engine: {engine_id}")
    except Exception as e:
        print(f"Note: could not delete Policy Engine yet ({e}). "
              f"It will be removed when the CFN stack is deleted (Step 5).")
else:
    print("Policy Engine not found (already deleted)")

## Step 2c: Delete OAuth2 Credential Provider

In [ ]:
# Delete OAuth2 Credential Provider
PROVIDER_NAME = "workshop-gateway-oauth"
providers = cp.list_oauth2_credential_providers().get("credentialProviders", [])
provider_match = [p for p in providers if p.get("name") == PROVIDER_NAME]

if provider_match:
    cp.delete_oauth2_credential_provider(name=PROVIDER_NAME)
    print(f"Deleted CredentialProvider: {PROVIDER_NAME}")
else:
    print("CredentialProvider not found (already deleted)")

## Step 2d: Delete EventBridge Rule (if still active)

In [ ]:
# The EventBridge rule `ac-registry-auto-review` is created and OWNED by the
# workshop-agentcore-stack (CloudFormation resource RegistryAutoReviewRule).
# Do NOT delete it here — deleting a stack-managed resource puts the stack into
# drift and breaks any later re-run of the registry steps (the auto-approve flow
# stops firing). It is removed automatically when the CFN stack is torn down
# (optional Step 5 below). This cell is intentionally a no-op.
RULE_NAME = "ac-registry-auto-review"
print(f"Leaving '{RULE_NAME}' in place — it is CloudFormation-managed by "
      f"workshop-agentcore-stack and is removed with the stack.")

## Step 3: Delete Registry Records

In [ ]:
# SDK workaround
def list_records_with_ids(client, registry_id, **kwargs):
    import json as _json
    original = client._endpoint.make_request
    raw = {}
    def capture(op, req):
        result = original(op, req)
        raw["data"] = _json.loads(result[0].content.decode("utf-8"))
        return result
    client._endpoint.make_request = capture
    try:
        client.list_registry_records(registryId=registry_id, **kwargs)
    finally:
        client._endpoint.make_request = original
    return raw.get("data", {}).get("registryRecords", [])

if REGISTRY_ID:
    records = list_records_with_ids(cp, REGISTRY_ID)
    print(f"Found {len(records)} record(s) to delete")
    for r in records:
        try:
            cp.delete_registry_record(registryId=REGISTRY_ID, recordId=r["recordId"])
            print(f"  Deleted: {r.get('name', r['recordId'])}")
        except Exception as e:
            print(f"  Error deleting {r.get('name', '?')}: {e}")
else:
    print("No registry found — skipping record deletion")

## Step 4: Delete the Registry

In [ ]:
if REGISTRY_ID:
    try:
        cp.delete_registry(registryId=REGISTRY_ID)
        print(f"Deleted registry: {REGISTRY_ID}")
    except Exception as e:
        print(f"Error: {e}")
else:
    print("No registry to delete")

## Step 5: (Optional) Delete the CFN Stack

> **WARNING:** Only delete the stack if you are NOT continuing to Module 4. Module 4 depends on the Gateway, Cognito, and Lambda resources.

In [ ]:
# UNCOMMENT the lines below ONLY if you are done with the workshop
# cfn = boto3.client("cloudformation", region_name=region)
# cfn.delete_stack(StackName="workshop-agentcore-stack")
# print("Stack deletion initiated. This takes ~5 minutes.")

print("Stack deletion is commented out.")
print("Uncomment the code above if you want to delete the CFN stack.")

## Verification

In [ ]:
# Check registry is gone
registries = cp.list_registries().get("registries", [])
workshop_regs = [r for r in registries if r["name"] == "workshop-registry"]
print(f"Registry:            {'DELETED' if not workshop_regs else 'STILL EXISTS'}")

# Check guardrail is gone
guardrails = bedrock.list_guardrails(maxResults=100).get("guardrails", [])
workshop_guards = [g for g in guardrails if g["name"] == "workshop-tool-output-guardrail"]
print(f"Guardrail:           {'DELETED' if not workshop_guards else 'STILL EXISTS'}")

# Check Policy Engine is gone
engines = cp.list_policy_engines().get("policyEngines", [])
workshop_engines = [e for e in engines if e.get("name") == "workshop_policy_engine"]
print(f"Policy Engine:       {'DELETED' if not workshop_engines else 'STILL EXISTS'}")

# Check CredentialProvider is gone
providers = cp.list_oauth2_credential_providers().get("credentialProviders", [])
workshop_providers = [p for p in providers if p.get("name") == "workshop-gateway-oauth"]
print(f"CredentialProvider:  {'DELETED' if not workshop_providers else 'STILL EXISTS'}")

# The ac-registry-auto-review EventBridge rule is CloudFormation-managed and is
# intentionally left in place (removed with the stack), so we do NOT assert on it here.

# Check CFN stack
cfn = boto3.client("cloudformation", region_name=region)
try:
    stack = cfn.describe_stacks(StackName="workshop-agentcore-stack")["Stacks"][0]
    print(f"CFN Stack:           {stack['StackStatus']}")
except Exception:
    print("CFN Stack:           DELETED")

# Summarize: warn on any remaining runtime resources (CFN stack deletion is optional)
remaining = []
if workshop_regs:          remaining.append("registry")
if workshop_guards:        remaining.append("guardrail")
if workshop_engines:       remaining.append("policy-engine")
if workshop_providers:     remaining.append("credential-provider")
if remaining:
    print(f"\nStill present (re-run this notebook or clean up manually): {remaining}")
else:
    print("\nAll runtime Module 3b resources cleaned up.")

## Summary

| Resource | Cleanup Method | Status |
|---|---|---|
| Bedrock Guardrail | `delete-guardrail` | Deleted |
| Policy Engine + Cedar Policies | `delete-policy` + `delete-policy-engine` | Deleted |
| OAuth2 CredentialProvider | `delete-oauth2-credential-provider` | Deleted |
| EventBridge Rule (`ac-registry-auto-review`) | CloudFormation-managed | Left in place (removed with stack) |
| Registry Records | `delete-registry-record` (loop) | Deleted |
| Registry | `delete-registry` | Deleted |
| CFN Stack | `delete-stack` (optional) | Kept for Module 4 |

If you deleted the CFN stack, all Lambda functions, Cognito resources, Gateway, IAM roles, WorkloadIdentity, the auto-review Lambda, and the `ac-registry-auto-review` EventBridge rule are also cleaned up.

**Module 3 complete!** Proceed to Module 4: Build Your First Agent.